In [ ]:
import streamlit as st
from docx import Document
from docx2pdf import convert
import tempfile
import datetime
import os

# --- Utilities to replace placeholders robustly ---
def replace_in_paragraph(paragraph, replacements):
    """
    Rebuild the paragraph text from runs, replace placeholders, then wipe runs and
    put back a single run with the replaced text (keeps paragraph-level formatting).
    """
    full_text = "".join(run.text for run in paragraph.runs)
    new_text = full_text
    changed = False
    for key, val in replacements.items():
        if key in new_text:
            new_text = new_text.replace(key, val)
            changed = True
    if changed:
        # Clear existing runs
        for run in paragraph.runs:
            run.text = ""
        # Add one run with the new text
        paragraph.add_run(new_text)


def replace_in_cell(cell, replacements):
    for p in cell.paragraphs:
        replace_in_paragraph(p, replacements)
    # If there are nested tables
    for table in cell.tables:
        replace_in_table(table, replacements)


def replace_in_table(table, replacements):
    for row in table.rows:
        for cell in row.cells:
            replace_in_cell(cell, replacements)


def replace_in_header_footer(header_or_footer, replacements):
    # paragraphs
    for p in header_or_footer.paragraphs:
        replace_in_paragraph(p, replacements)
    # tables
    for table in header_or_footer.tables:
        replace_in_table(table, replacements)


def replace_all(doc, replacements):
    # Main document paragraphs
    for p in doc.paragraphs:
        replace_in_paragraph(p, replacements)

    # Tables in body
    for table in doc.tables:
        replace_in_table(table, replacements)

    # Headers and footers for all sections
    for section in doc.sections:
        header = section.header
        footer = section.footer
        replace_in_header_footer(header, replacements)
        replace_in_header_footer(footer, replacements)

    return doc


# --- Generate the letter ---
def generate_letter(template_path, context):
    doc = Document(template_path)
    doc = replace_all(doc, context)
    return doc


# --- Save and convert to PDF ---
def save_and_convert_to_pdf(doc, student_name):
    temp_dir = tempfile.mkdtemp()
    safe_name = "".join(c for c in student_name if c.isalnum() or c in (" ", "_", "-")).rstrip()
    docx_path = os.path.join(temp_dir, f"{safe_name}.docx")
    pdf_path = os.path.join(temp_dir, f"{safe_name}.pdf")
    doc.save(docx_path)
    convert(docx_path, pdf_path)
    return docx_path, pdf_path


# --- Extract plain text preview ---
def get_text_preview(doc):
    parts = []
    for p in doc.paragraphs:
        text = p.text.strip()
        if text:
            parts.append(text)
    return "\n\n".join(parts)


# --- Streamlit UI ---
st.set_page_config(page_title="Recommendation Letter Generator", layout="centered")
st.title("🎓 Recommendation Letter Generator (robust placeholder replacement)")
st.write("This version replaces placeholders in body, tables, headers, and footers. "
         "If placeholders are inside textboxes/shapes they may still be missed — see notes below.")

with st.form("form"):
    full_name = st.text_input("Full Name of Student")
    gender = st.selectbox("Gender", ["Male", "Female"])
    university = st.text_input("University Applying To")
    project_topic = st.text_input("Final Year Project Topic")
    grad_class = st.text_input("Graduating Class (e.g., First Class, Second Upper)")
    cwa = st.text_input("Cumulative Weighted Average (CWA)")
    year = st.text_input("Year You Began Teaching the Student")
    submitted = st.form_submit_button("Generate Letter")

if submitted:
    required = [full_name, gender, university, project_topic, grad_class, cwa, year]
    if not all(required):
        st.warning("Please fill in all fields.")
    else:
        current_date = datetime.date.today().strftime("%B %d, %Y")
        replacements = {
            "[Full Name]": full_name,
            "[University Applying To]": university,
            "[Project Topic]": project_topic,
            "[Graduating Class]": grad_class,
            "[CWA]": cwa,
            "[Year]": year,
            "[Date]": current_date
        }

        template_file = "Male.docx" if gender == "Male" else "Female.docx"

        try:
            doc = generate_letter(template_file, replacements)
        except Exception as e:
            st.error(f"Failed to open/modify template: {e}")
            raise

        # Save and convert
        docx_path, pdf_path = save_and_convert_to_pdf(doc, full_name.replace(" ", "_"))

        st.success("Letter generated.")
        preview = get_text_preview(doc)
        st.markdown("### Preview (plain text):")
        st.text_area("", preview, height=500)

        st.markdown("### Downloads")
        with open(docx_path, "rb") as f:
            st.download_button("Download DOCX", f, file_name=os.path.basename(docx_path))
        with open(pdf_path, "rb") as f:
            st.download_button("Download PDF", f, file_name=os.path.basename(pdf_path))

        st.info("Note: Downloads preserve the original letterhead and signature from the template.")
